# SporeDB Demo: Fermentation Run Analysis

This notebook demonstrates the complete SporeDB workflow for analyzing
fermentation data from *Rhodosporidium toruloides* runs on different
carbon sources.

**What you'll learn:**
1. Load and explore fermentation data
2. Automatically detect growth phases
3. Align runs for comparison
4. Compare performance across conditions
5. Visualize multi-run overlays

**Prerequisites:** Run `sporedb demo load` before starting this notebook.

In [ ]:
from pathlib import Path
from sporedb import SporeDB

# Connect to the demo database
DEMO_DATA = Path.home() / ".sporedb" / "demo_data"
if not DEMO_DATA.exists():
    raise RuntimeError(
        "Demo data not found. Run 'sporedb demo load' first.\n"
        "Install SporeDB CLI: pip install sporedb"
    )

db = SporeDB(str(DEMO_DATA))
batches = db.list_batches()
print(f"Found {len(batches)} fermentation runs:")
for b in batches:
    print(f"  - {b.name} (strain: {b.metadata.strain})")

## Step 1: Explore the Fermentation Data

Each run contains time-series telemetry from a bioreactor:
- **Dissolved Oxygen (DO):** Oxygen consumption indicates metabolic activity
- **pH:** Tracks acid/base production during fermentation
- **Temperature:** Controlled at 28 deg C by the bioreactor PID loop
- **Biomass (OD600):** Optical density measures cell growth
- **Agitation:** RPM increases to maintain oxygen transfer
- **Feeding Rate:** Carbon source addition in fed-batch mode

SporeDB stores telemetry in a normalized format: each row records a
single measurement with its timestamp, variable name, value, and unit.

In [ ]:
import pandas as pd

# Get telemetry for the glucose run
glucose_batch = [b for b in batches if "glucose" in b.name.lower()][0]
df = db.get_telemetry(glucose_batch.batch_id)
print(f"Run: {glucose_batch.name}")
print(f"Variables: {sorted(df['variable'].unique())}")
print(f"Data points: {len(df)}")
print(f"Time range: {df['ts'].min()} to {df['ts'].max()}")
df.head(10)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pivot the narrow-format telemetry to wide format for plotting
wide = df.pivot_table(index="ts", columns="variable", values="value", aggfunc="first")

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=["Cell Growth (Biomass / OD600)", "Dissolved Oxygen (%)", "pH"],
)

fig.add_trace(
    go.Scatter(x=wide.index, y=wide["biomass"], name="Biomass",
               line=dict(color="#1f77b4")),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=wide.index, y=wide["dissolved_oxygen"], name="DO",
               line=dict(color="#ff7f0e")),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=wide.index, y=wide["ph"], name="pH",
               line=dict(color="#2ca02c")),
    row=3, col=1,
)

fig.update_layout(
    height=600, template="plotly_white",
    title_text=f"Raw Telemetry: {glucose_batch.name}",
    hovermode="x unified", showlegend=False,
)
fig.show()

## Step 2: Automatic Phase Detection

Fermentation runs go through distinct biological phases:
- **Lag phase:** Cells adapt to the medium (low growth)
- **Exponential phase:** Rapid cell division (DO drops, OD rises)
- **Stationary phase:** Growth plateaus (nutrients depleted)
- **Decline phase:** Cell viability decreasing

SporeDB uses the PELT changepoint detection algorithm to automatically
identify these transitions from the telemetry data.

In [ ]:
phases = db.detect_phases(glucose_batch.batch_id, signal="biomass")
print(f"Detected {len(phases)} phases in {glucose_batch.name}:\n")
for i, phase in enumerate(phases):
    print(f"  Phase {i+1}: {phase.phase_type.value}")
    print(f"    Start: {phase.start_ts}")
    print(f"    End:   {phase.end_ts}")
    print(f"    Confidence: {phase.confidence:.2f}")
    print()

## Step 3: Align Runs for Comparison

Different runs start at different times and may have different sampling
rates. SporeDB's `align()` function normalizes runs to a common timeline
by anchoring on the exponential phase boundary, making direct comparison
possible.

We'll align all three runs -- glucose, xylose, and hydrolysate feedstocks.

In [ ]:
batch_ids = [b.batch_id for b in batches]
aligned_df = db.align(batch_ids, signal="biomass")
print(f"Aligned DataFrame shape: {aligned_df.shape}")
print(f"Columns: {list(aligned_df.columns)}")
aligned_df.head()

## Step 4: Compare Across Conditions

Now we can compare how *R. toruloides* performs on different carbon sources.
Key questions for a fermentation scientist:
- Which feedstock gives the highest cell density?
- How does dissolved oxygen profile differ?
- Which condition reaches stationary phase fastest?

In [ ]:
print("Batch Performance Comparison\n")
print(f"{'Run':<35} {'Max Biomass':>12} {'Min DO':>10} {'Phases':>8}")
print("-" * 70)
for batch in batches:
    telem = db.get_telemetry(batch.batch_id)
    # Extract per-variable statistics from narrow-format data
    biomass_data = telem[telem["variable"] == "biomass"]["value"]
    do_data = telem[telem["variable"] == "dissolved_oxygen"]["value"]
    phases = db.detect_phases(batch.batch_id, signal="biomass")
    max_biomass = biomass_data.max() if len(biomass_data) > 0 else float("nan")
    min_do = do_data.min() if len(do_data) > 0 else float("nan")
    print(f"{batch.name:<35} {max_biomass:>12.2f} {min_do:>10.2f} {len(phases):>8}")

## Step 5: Visualize Multi-Run Overlay

SporeDB's `overlay_runs()` creates an interactive comparison plot.
Each variable gets its own subplot with all runs overlaid,
color-coded by batch. Phase boundaries are shown as vertical dashed lines.

In [ ]:
from sporedb.viz import overlay_runs

fig = overlay_runs(
    db, batch_ids,
    variables=["biomass", "dissolved_oxygen", "ph"],
    show_phases=True,
)
fig.show()

## What's Next?

You've just completed the full SporeDB scientist workflow:
1. **Loaded** real fermentation telemetry data
2. **Detected** growth phases automatically using PELT algorithm
3. **Aligned** runs across different conditions
4. **Compared** performance metrics across feedstocks
5. **Visualized** multi-run overlays with phase annotations

### Learn More
- [SporeDB Documentation](https://ranjanrishikesh.github.io/SporeDB/)
- [API Reference](https://ranjanrishikesh.github.io/SporeDB/reference/sporedb/)
- [Configuration Guide](https://ranjanrishikesh.github.io/SporeDB/configuration/)
- [GitHub Repository](https://github.com/ranjanrishikesh/SporeDB)

In [ ]:
db.close()